# Surface-Based Source Spaces

The Source Space is a distributed dipole model. Brain sources are approximated by dipoles and placed uniformly along the brain surface (more specifically, the white surface — the boundary between gray and white matter). The number of dipoles is adjustable based on the `spacing`. We then eliminate any dipole that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

<img src="../../img/sourcespace.png">

## 📦 Imports

In [1]:
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**

**Anatomy**
- FREESURFER reconstruction folder.
    - `white`: white surface
- `ctx`: Cortical Mask.

**Transformations**
- `tkras2ras`: Affine transform (freesurfer ras -> T1 ras)

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [8]:
# ------------- Parameters ---------------
fconfig = "../../data/config.yml"
sub     = "intra-12"
# ----------------------------------------

reader    = mm.io.TutoReader(fconfig)
filtering = "nifti"

try:
    data = reader(sub, "ctx", "tkras2ras")
except FileNotFoundError:
    data = reader(sub, "vertex-ctx")
    filtering = "vertex-id"

DATA SUMMARY
Dataset: tuto
Subject: intra-12
Task: None
Run: None
Retrieved objects:
- vertex-ctx
Missing objects:


## ⚙️ Build Source Spaces

MNE selects dipoles on the white matter surface by uniformly downsampling the white mesh. The parameter “octX” defines the downsampling level of the triangular mesh.


📤 **<font color=pink>Outputs</font>**
- `source_space`: MNE object containing dipole locations and dipole distances to each other.

🎚️ **<font color=pink>Setup source space</font>**:
- `spacing`: Spacing in [mm] between dipoles or mesh downsampling level.
- `n_jobs`: Number of parrallel jobs

In [3]:
# ------------ Parameters -------------
# source space
spacing = "oct6"

# computing
n_jobs = 10
# -------------------------------------


# source space
source_space = mne.setup_source_space(
    subject      = f"sub-{data['sub']}",
    spacing      = spacing,
    surface      = "white",
    add_dist     = True,                    # geodesic distances between sources
    subjects_dir = reader.config["FS_DIR"],
    n_jobs       = n_jobs,
    verbose      = "warning",
)
    
# gray matter constraint
if filtering == "nifti":
    source_space = mm.head.filter_dipoles_by_nifti_mask(source_space, data["ctx"], data["tkras2ras"])
else:
    source_space = mm.head.filter_dipoles_by_id_list(source_space, data["vertex-ctx"])

print("\033[92mSUMMARY\033[0m")
for src in source_space:
    print(f"Source space id: {src['id']}")
    print(f"Type: {src['type']}")
    print(f"Number of sources: {src['nuse']}\n")

SUMMARY
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 3821

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 3826



📊 **Source space** <br>

In [6]:
if filtering == "nifti":
    mm.viz.plot_brain(source_space, show_src=True, overlay=data["ctx"], tkras2ras=data["tkras2ras"], overlay_color="gray", overlay_opacity=0.5).show()
else:
    mm.viz.plot_brain(source_space, mesh_params={"show_src":True}).show()

Widget(value='<iframe src="http://localhost:36583/index.html?ui=P_0x7112b7a1e390_0&reconnect=auto" class="pyvi…

💾 **Save Source Space**

🎚️ **<font color=pink>Save</font>**:
- `overwrite`: if True, overwrites previously saved data.

In [17]:
# ------------ Parameters -------------
overwrite = True

file = mm.io.get_bidspath(
    root        = reader.config["MM_DIR"],
    subject     = sub,
    datatype    = "src",
    task        = None,
    session     = None,
    description = None,
)
# -------------------------------------
file.parent.mkdir(parents=True, exist_ok=True)

source_space.save(file, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved surface-based source space for subject {data['sub']}\033[0m")

Successfully saved surface-based source space for subject intra-12


In [15]:
data2 = reader(sub, "src-raw")

len(data2["src-raw"][0]["pinfo"])

DATA SUMMARY
Dataset: tuto
Subject: intra-12
Task: None
Run: None
Retrieved objects:
- src-raw
Missing objects:


4098

In [54]:
import numpy as np

vertno  = np.array(data2["src-raw"][0]["vertno"])
nearest = data2["src-raw"][0]["nearest"].copy()

valid = np.isin(nearest, vertno)
#nearest[~valid] = -1

In [55]:
np.all(np.sort(np.unique(nearest))[1:]==np.sort(vertno))

ValueError: operands could not be broadcast together with shapes (4097,) (3821,) 

In [50]:
np.sort(vertno)

array([    73,    102,    120, ..., 145468, 145554, 145701])

In [44]:
np.unique(nearest)[1:]

array([    73,    102,    120, ..., 145468, 145554, 145701], dtype=int32)